# nanogpt-tinystories — one-click training on a free Colab T4

Train a ~30M-parameter GPT **from scratch** on TinyStories until it writes little
coherent stories. End to end this takes **~30–40 minutes** on a free T4:

1. check the GPU  2. get the code + deps  3. tokenize TinyStories
4. train 5000 iters (→ ~1.7 val loss)  5. generate stories
6. watch the learning progression  7. plot the loss curve

> **Before you start:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU`.
> Everything else runs top-to-bottom (`Runtime → Run all`).

## 0. Check the GPU
Confirm a GPU is attached (you want to see a Tesla T4).

In [ ]:
!nvidia-smi

## 1. Get the code and install dependencies

This notebook runs the project's own `src/` modules, so we clone the repo.
**Push this project to GitHub first**, then set `REPO_URL` to your repo below.

Colab already ships `torch`, `numpy`, `tqdm`, and `matplotlib`, so we only add the
two it lacks: `tiktoken` (the GPT-2 tokenizer) and `datasets` (to fetch
TinyStories).

In [ ]:
# EDIT THIS to your pushed repository, then run the cell.
REPO_URL = "https://github.com/<your-username>/nanogpt-tinystories.git"

!git clone $REPO_URL
%cd nanogpt-tinystories
!pip install -q tiktoken datasets

## 2. Build the dataset

Download TinyStories and tokenize it into `data/train.bin` / `data/val.bin`
(~474M / ~4.8M tokens). First run downloads ~2GB and takes a few minutes; it's
cached afterward.

In [ ]:
!python -m src.data

## 3. Train the full run (~30 min on a T4)

Uses the proven config from `src/config.py`: 6 layers, 6 heads, 384-dim, 256
context, batch 32, 5000 iters, AdamW with warmup+cosine LR. On the T4 we train in
float16 (the T4 has no native bfloat16).

`--sample-every 1000` writes a generated story to `samples/progression.md` every
1000 steps, so you can literally watch it learn (gibberish → words → stories).
Validation loss should fall from ~10.8 toward **~1.7**.

In [ ]:
!python -m src.train --sample-every 1000

## 4. Generate stories
Sample from the best checkpoint. Try different prompts, temperatures, and top-k.

In [ ]:
!python -m src.generate --prompt "Once upon a time" --num-samples 3 --temperature 0.8 --top-k 200

## 5. See the learning progression
The samples captured during training — from nonsense to coherent little stories.

In [ ]:
from pathlib import Path
print(Path("samples/progression.md").read_text(encoding="utf-8"))

## 6. Plot the loss curve
Train vs. validation loss over the run (saved to `samples/loss_curve.png`).

In [ ]:
import torch, matplotlib.pyplot as plt

ckpt = torch.load("checkpoints/best.pt", map_location="cpu", weights_only=False)
hist = ckpt["history"]
iters = [r["iter"] for r in hist]

plt.figure(figsize=(7, 4))
plt.plot(iters, [r["train"] for r in hist], label="train", marker="o", ms=3)
plt.plot(iters, [r["val"] for r in hist], label="val", marker="o", ms=3)
plt.xlabel("iteration"); plt.ylabel("cross-entropy loss")
plt.title("TinyStories GPT — training curve"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig("samples/loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()
print("final val loss:", ckpt["best_val_loss"])

## 7. (Optional) Download your trained model and samples

In [ ]:
from google.colab import files
files.download("checkpoints/best.pt")   # ~360 MB (weights + optimizer state)
# files.download("samples/loss_curve.png")